In this cell, the student health and grades dataset is downloaded using KaggleHub and the file path is printed.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("srisyra02/ai-and-social-media-impact-student-health-and-grades")

print("Path to dataset files:", path)

The pandas and numpy libraries required for data analysis and numerical operations are imported.

In [ ]:
import pandas as pd
import numpy as np

The downloaded CSV file is read with pandas and loaded into a DataFrame.

In [ ]:
df = pd.read_csv("/kaggle/input/ai-and-social-media-impact-student-health-and-grades/AI_SocialMedia_Student_Dataset.csv")

Since the identifier is unnecessary for analysis, the Student_ID column is removed from the dataset.

In [ ]:
df.drop(columns=["Student_ID"], inplace=True)

The structure of the dataset, column types, and basic statistical summaries are examined.

In [ ]:
display(df.info())
df.describe()

The seaborn and matplotlib libraries are imported for visualization.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

Correlations between numerical variables are visualized with a heatmap.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.show()

Categorical variables are converted to numerical form using one-hot encoding.

In [ ]:
df = pd.get_dummies(df)

Features (X) are separated from the target variables (mental and physical health scores).

In [ ]:
X = df.drop(columns=["Mental_Health_Score", "Physical_Health_Score"])
y1 = df["Mental_Health_Score"]
y2 = df["Physical_Health_Score"]

The data is split into training and test sets, and the features are standardized.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train, X_test, y1_train, y1_test = train_test_split(
    X, y1, test_size=0.2, random_state=42
)
_, _, y2_train, y2_test = train_test_split(
    X, y2, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

The features in the training set are displayed.

In [ ]:
X_train

Linear regression and Random Forest models are trained and evaluated on the test set using MAE and R².

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("--- Mental Health Score (y1) Prediction Results ---")
for name, model in models.items():
    model.fit(X_train_scaled, y1_train)
    preds = model.predict(X_test_scaled)
    mae = mean_absolute_error(y1_test, preds)
    r2 = r2_score(y1_test, preds)
    print(f"{name} -> MAE: {mae:.4f}, R2 Score: {r2:.4f}")

print("\n--- Physical Health Score (y2) Prediction Results ---")
for name, model in models.items():
    model.fit(X_train_scaled, y2_train)
    preds = model.predict(X_test_scaled)
    mae = mean_absolute_error(y2_test, preds)
    r2 = r2_score(y2_test, preds)
    print(f"{name} -> MAE: {mae:.4f}, R2 Score: {r2:.4f}")

Separate linear regression and Random Forest models are defined and trained for the mental and physical health scores.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lr_model_y1 = LinearRegression()
lr_model_y2 = LinearRegression()
rf_model_y1 = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_y2 = RandomForestRegressor(n_estimators=100, random_state=42)

lr_model_y1.fit(X_train_scaled, y1_train)
lr_model_y2.fit(X_train_scaled, y2_train)
rf_model_y1.fit(X_train_scaled, y1_train)
rf_model_y2.fit(X_train_scaled, y2_train)

Health score predictions are made with the models for two example student profiles and the results are printed.

In [ ]:
new_data = pd.DataFrame([
    {
        "Age": 23,
        "Daily_Social_Media_Hours": 8.0,
        "Daily_AI_Tool_Usage_Hours": 0.5,
        "Sleep_Hours": 11.0,
        "Physical_Activity_Hours": 0.5,
        "Gender_Female": False,
        "Gender_Male": True,
        "Gender_Non-binary": False,
        "Education_Level_College": False,
        "Education_Level_High School": False,
        "Education_Level_University": True
    },
    {
        "Age": 23,
        "Daily_Social_Media_Hours": 4.5,
        "Daily_AI_Tool_Usage_Hours": 3.0,
        "Sleep_Hours": 8.0,
        "Physical_Activity_Hours": 1.2,
        "Gender_Female": False,
        "Gender_Male": True,
        "Gender_Non-binary": False,
        "Education_Level_College": False,
        "Education_Level_High School": False,
        "Education_Level_University": True
    }
])

new_data = new_data[X.columns]

new_data_scaled = scaler.transform(new_data)

pred_lr_y1 = lr_model_y1.predict(new_data_scaled)
pred_lr_y2 = lr_model_y2.predict(new_data_scaled)
pred_rf_y1 = rf_model_y1.predict(new_data_scaled)
pred_rf_y2 = rf_model_y2.predict(new_data_scaled)

print("=== PROFILE 1 PREDICTIONS (Social Media: 6h, AI: 1h, Sleep: 10h, Activity: 3h) ===")
print(f"Linear Regression -> Mental: {pred_lr_y1[0]:.2f} | Physical: {pred_lr_y2[0]:.2f}")
print(f"Random Forest     -> Mental: {pred_rf_y1[0]:.2f} | Physical: {pred_rf_y2[0]:.2f}")

print("\n=== PROFILE 2 PREDICTIONS (Social Media: 3h, AI: 3h, Sleep: 7h, Activity: 9h) ===")
print(f"Linear Regression -> Mental: {pred_lr_y1[1]:.2f} | Physical: {pred_lr_y2[1]:.2f}")
print(f"Random Forest     -> Mental: {pred_rf_y1[1]:.2f} | Physical: {pred_rf_y2[1]:2f}")

The predicted profile scores are visualized with a scatter plot over the actual dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))

sns.scatterplot(
    x=df['Mental_Health_Score'],
    y=df['Physical_Health_Score'],
    alpha=0.15,
    color='gray',
    label='General Student Distribution'
)

plt.scatter(
    pred_lr_y1[0], pred_lr_y2[0],
    color='blue', marker='o', s=100, edgecolor='black', zorder=5,
    label=f'Profile 1 - Linear Reg. (Mental: {pred_lr_y1[0]:.1f}, Physical: {pred_lr_y2[0]:.1f})'
)
plt.scatter(
    pred_rf_y1[0], pred_rf_y2[0],
    color='cyan', marker='o', s=100, edgecolor='black', zorder=5,
    label=f'Profile 1 - Random Forest (Mental: {pred_rf_y1[0]:.1f}, Physical: {pred_rf_y2[0]:.1f})'
)

plt.scatter(
    pred_lr_y1[1], pred_lr_y2[1],
    color='red', marker='o', s=100, edgecolor='black', zorder=5,
    label=f'Profile 2 - Linear Reg. (Mental: {pred_lr_y1[1]:.1f}, Physical: {pred_lr_y2[1]:.1f})'
)
plt.scatter(
    pred_rf_y1[1], pred_rf_y2[1],
    color='orange', marker='o', s=100, edgecolor='black', zorder=5,
    label=f'Profile 2 - Random Forest (Mental: {pred_rf_y1[1]:.1f}, Physical: {pred_rf_y2[1]:.1f})'
)

plt.title('Distribution of Predicted Profile Scores over the Actual Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Mental Health Score', fontsize=12)
plt.ylabel('Physical Health Score', fontsize=12)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

This cell is left blank.